# Homework 1 — Intro and Data Sources

Stock Markets Analytics Zoomcamp, 2026 cohort.

Assignment: [cohorts/2026/homework1.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md)

In [ ]:
import io

import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_rows", 40)

## Question 1 — S&P 500 additions

Which year had the highest number of additions starting from 2020?

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}
html = requests.get(url, headers=headers, timeout=30).text
sp500 = pd.read_html(io.StringIO(html))[0]
sp500["added_year"] = pd.to_datetime(sp500["Date added"], errors="coerce").dt.year

counts = sp500.loc[sp500["added_year"] >= 2020, "added_year"].value_counts().sort_index()
counts

**Answer: 2025** (18 current constituents were added that year; next is 2024 with 16).

Additional: 224 current S&P 500 stocks have been in the index for more than 20 years (added before 13 September 2006).

## Question 2 — Index YTD returns as of 21 August 2026

How many of the 10 non-US indexes beat the S&P 500 year-to-date?

In [ ]:
indexes = {
    "United States (S&P 500)": "^GSPC",
    "China (Shanghai Composite)": "000001.SS",
    "Hong Kong (Hang Seng)": "^HSI",
    "Australia (S&P/ASX 200)": "^AXJO",
    "India (Nifty 50)": "^NSEI",
    "Canada (S&P/TSX)": "^GSPTSE",
    "Germany (DAX)": "^GDAXI",
    "United Kingdom (FTSE 100)": "^FTSE",
    "Japan (Nikkei 225)": "^N225",
    "Mexico (IPC)": "^MXX",
    "Brazil (Ibovespa)": "^BVSP",
}

# yfinance end is exclusive, so 2026-08-22 includes 21 August.
data = yf.download(
    list(indexes.values()),
    start="2026-01-01",
    end="2026-08-22",
    auto_adjust=True,
    progress=False,
)
close = data["Close"]

ytd = {}
for name, ticker in indexes.items():
    s = close[ticker].dropna()
    ytd[name] = s.iloc[-1] / s.iloc[0] - 1

ytd = pd.Series(ytd).sort_values(ascending=False)
us = ytd["United States (S&P 500)"]
print((ytd * 100).round(2))
print(f"\nS&P 500 YTD: {us*100:.2f}%")
print("Indexes beating US:", int((ytd.drop(labels=["United States (S&P 500)"]) > us).sum()))

**Answer: 2**

S&P 500 YTD ≈ 11.9%. Only Japan (Nikkei 225, ≈ 27.4%) and Canada (S&P/TSX, ≈ 14.9%) did better.

Additional: the same two also beat the S&P 500 over 3 and 5 years; over 10 years only Japan did. The 2026 YTD gap is not a one-off for those two markets, but most of the other eight still lag the US over longer windows.

## Question 3 — S&P 500 correction drawdowns

Median drawdown (%) of corrections of at least 5% from the prior all-time high.

In [ ]:
raw = yf.download("^GSPC", start="1950-01-01", auto_adjust=True, progress=False)
spx = raw["Close"]
if isinstance(spx, pd.DataFrame):
    spx = spx.iloc[:, 0]
spx = spx.dropna()

prev_max = spx.shift(1).cummax()
is_ath = spx > prev_max
is_ath.iloc[0] = True
ath_dates = spx.index[is_ath]

rows = []
for peak_date, next_ath in zip(ath_dates[:-1], ath_dates[1:]):
    window = spx.loc[peak_date:next_ath].iloc[1:-1]
    if window.empty:
        continue
    trough_date = window.idxmin()
    peak_px = float(spx.loc[peak_date])
    trough_px = float(window.min())
    rows.append(
        {
            "peak": peak_date.date(),
            "trough": trough_date.date(),
            "drawdown_pct": (peak_px - trough_px) / peak_px * 100,
            "days": (trough_date - peak_date).days,
        }
    )

corr = pd.DataFrame(rows)
corr = corr[corr["drawdown_pct"] >= 5]
print("corrections >= 5%:", len(corr))
print("drawdown percentiles\n", corr["drawdown_pct"].quantile([0.25, 0.50, 0.75]).round(2))
print("duration percentiles\n", corr["days"].quantile([0.25, 0.50, 0.75]).round(1))
print("\nlargest 10")
print(
    corr.sort_values("drawdown_pct", ascending=False)
    .head(10)[["peak", "trough", "drawdown_pct", "days"]]
    .to_string(index=False)
)

**Answer: 8**

Median drawdown among 74 corrections of at least 5% is about 7.99%. Quartiles: 6.2% / 8.0% / 14.0%. Median duration is about 40–41 days.

The ten largest drawdowns match the homework hint (GFC 56.8%, dot-com 49.1%, 1973–74 48.2%, …).

## Question 4 — Amazon earnings surprises

Median 2-day percentage change after positive earnings surprises. Day 2 is the announcement date; return is `Close_Day3 / Close_Day1 - 1`.

In [ ]:
earn = yf.Ticker("AMZN").get_earnings_dates()
prices = yf.download("AMZN", start="2018-01-01", auto_adjust=True, progress=False)
amzn = prices["Close"]
if isinstance(amzn, pd.DataFrame):
    amzn = amzn.iloc[:, 0]
amzn = amzn.dropna()

ret2 = amzn.shift(-1) / amzn.shift(1) - 1

idx = pd.to_datetime(earn.index).tz_convert("America/New_York").tz_localize(None)
earn = earn.copy()
earn.index = idx.normalize()
earn = earn.join(ret2.rename("ret2"), how="left")

pos = earn[earn["Surprise(%)"] > 0]
print(earn[["EPS Estimate", "Reported EPS", "Surprise(%)", "ret2"]])
print("\npositive surprises:", len(pos))
print("median 2-day return %:", round(float(pos["ret2"].median()) * 100, 2))
print(earn[["Surprise(%)", "ret2"]].dropna().corr())

**Answer: 0.35**

There are 25 earnings rows from 2020-10-29 (one future date with no reported EPS). Twenty of the historical prints were positive surprises. The median 2-day return on those days is 0.35%.

Additional: correlation between surprise magnitude and the 2-day return is only about 0.22 — a weak positive link. A few large beats (for example 2026-07-30) moved the stock a lot, but several other beats were sold. The sample is too small to split bull vs bear cleanly, but the 2022 prints already show that a beat does not guarantee a bounce (2022-10-27 was −10.6%).

## Question 5 — Capstone idea

I want to build a **20-trading-day post-earnings drift model for large-cap US stocks** (S&P 500 names, starting with mega-cap tech and then the full index).

The question I want the model to answer is: *after a quarterly earnings print, is the stock more likely to outperform SPY over the next 20 sessions, and should I buy the dip or fade the pop?* Homework 4 already shows that a positive surprise is not enough — Amazon’s median 2-day reaction is only +0.35%, and the surprise-to-return correlation is weak. Homework 3 shows that most S&P 500 corrections are short and shallow (median ~8% and ~40 days), so a strategy that blindly “buys every beat” will get run over in those windows.

Planned features:
- earnings surprise %, revenue surprise, and whether guidance was raised
- 2-day announcement return (the same construction as Q4)
- market regime: distance from the S&P 500 all-time high, VIX level, and whether we are inside a ≥5% correction
- rates and dollar: 10-year yield change and DXY return over the prior month
- sector relative strength (stock vs its GICS sector ETF)

The label is 20-day excess return vs SPY. I would train a simple classifier or gradient-boosted ranker (no high-frequency data), then turn the scores into a long-only or long/flat rule and simulate it with transaction costs. If the model works, I would automate a weekly refresh of the earnings calendar and scores.

## Question 6 — Extra metrics

I pulled a few extra Yahoo Finance series that I would actually use in that project. All of them come from `yfinance.download` the same way as the homework indexes.

In [ ]:
extra = {
    "VIX": "^VIX",
    "US 10Y yield": "^TNX",
    "US Dollar Index": "DX-Y.NYB",
    "SPY": "SPY",
    "Tech (XLK)": "XLK",
    "Financials (XLF)": "XLF",
    "High yield (HYG)": "HYG",
    "Long Treasuries (TLT)": "TLT",
}
px = yf.download(list(extra.values()), start="2021-01-01", end="2026-08-22", auto_adjust=True, progress=False)["Close"]

def ytd_ret(s):
    s = s.dropna()
    y = s.loc["2026-01-01":]
    return float(y.iloc[-1] / y.iloc[0] - 1)

print(f"as of {px.index.max().date()}")
for name, ticker in extra.items():
    s = px[ticker].dropna()
    print(f"{name:22} {ticker:10} last={s.iloc[-1]:8.2f}  YTD={ytd_ret(s)*100:6.1f}%")

Why these help the project:

1. **VIX (`^VIX`)** — regime filter. 2026 YTD the VIX has been relatively calm (median ~18, last print ~15) versus 2022 (median ~25). I would use VIX level and 20-day change to mark “risk-on vs correction” before taking an earnings trade.
2. **US 10-year yield (`^TNX`)** — discount-rate shock. The 10-year is up about 13% YTD in 2026 (yield near 4.74). Growth names like AMZN are more sensitive to this than the surprise itself; I would include the 20-day yield change as a feature.
3. **US Dollar Index (`DX-Y.NYB`)** — international transmission. Homework 2 already showed most non-US indexes lagging the S&P 500. A rising dollar is one reason; DXY was roughly flat YTD (+0.4%) as of 21 Aug 2026, so 2026 outperformance of Japan/Canada is not just a weak-dollar story.
4. **Sector ETFs (`XLK`, `XLF`)** — relative strength. XLK is +27% YTD vs XLF +5.5% and SPY +12.7%. A beat in a strong sector is a different trade from a beat in a weak one.
5. **Credit (`HYG`) and duration (`TLT`)** — risk appetite and the rates overlay. HYG is only +2.1% YTD while TLT is −3.2%. I would use HYG−TLT or HYG vs SPY as a simple credit-risk feature.

Python retrieval is the same pattern as the homework:

```python
import yfinance as yf
px = yf.download(["^VIX", "^TNX", "DX-Y.NYB", "XLK", "XLF", "HYG", "TLT"],
                 start="2015-01-01", auto_adjust=True)
```

For the earnings calendar itself I would keep using `yf.Ticker(symbol).get_earnings_dates()`, and for the S&P 500 universe I would reuse the Wikipedia scrape from Question 1.